<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Case_Study_5_LangChain_Banking_Tools_Schema_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 5: Tool Integration with Schema Validation
## LangChain Banking Tool Agent

### Objective

Build a banking agent that can dynamically select and call:

- `get_credit_score(user_id)`
- `get_transaction_history(user_id)`

The implementation must:

1. define strict tool input and output schemas;
2. reject invalid tool inputs;
3. allow the model to select tools dynamically;
4. validate tool outputs before returning them to the model; and
5. handle tool failures without crashing the agent.

### Framework

- **LangChain** — model/tool integration
- **LangChain OpenAI** — OpenAI chat model
- **Pydantic** — strict input/output validation
- **Plain Python agent loop** — explicit and inspectable tool execution

> **Safety and privacy notice:** This notebook uses only fictional users and mock banking data. Never place real banking credentials, account numbers, authentication secrets, or customer records in a shared notebook.

## Architecture

```text
User request
    │
    ▼
OpenAI model with bound tool schemas
    │
    ├── No tool required ───────────────► Final response
    │
    └── Select one or more tools
                    │
                    ▼
            Validate tool input
                    │
          ┌─────────┴─────────┐
          │ Valid             │ Invalid
          ▼                   ▼
     Execute mock tool    Structured error
          │
          ▼
     Validate output
          │
          ├── Valid output ───► ToolMessage
          └── Invalid output ─► Structured error
                                  │
                                  ▼
                          Model produces response
```

The language model chooses **which** tool to use. Python remains responsible for validation, execution, error handling, and loop limits.

## 1. Install dependencies

In [1]:
!pip -q install --upgrade langchain langchain-openai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


## 2. Configure the OpenAI API key

The API key is entered securely with `getpass` and is not printed.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured.")

Enter your OpenAI API key: ··········
OpenAI API key configured.


## 3. Imports and model configuration

In [3]:
import json
import logging
from datetime import date
from importlib.metadata import version
from pprint import pprint
from typing import (
    Annotated,
    Any,
    Dict,
    List,
    Literal,
    Optional,
    Set,
    Union,
)

import pandas as pd
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import BaseTool, tool
from langchain_openai import ChatOpenAI
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    TypeAdapter,
    ValidationError,
    field_validator,
    model_validator,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)
logger = logging.getLogger("banking_tool_agent")

print("Model:", MODEL)
print("langchain:", version("langchain"))
print("langchain-openai:", version("langchain-openai"))
print("pydantic:", version("pydantic"))

Model: gpt-4o-mini
langchain: 1.3.13
langchain-openai: 1.3.5
pydantic: 2.13.4


## 4. Define strict input schemas

A valid fictional user ID must follow:

```text
USR-1234
```

The schemas:

- reject malformed IDs;
- strip whitespace;
- normalise IDs to uppercase; and
- reject unexpected fields.

In [4]:
USER_ID_PATTERN = r"^USR-[0-9]{4}$"


class StrictSchema(BaseModel):
    """Base class that rejects undeclared fields."""

    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
    )


class GetCreditScoreInput(StrictSchema):
    """Input schema for get_credit_score."""

    user_id: str = Field(
        min_length=8,
        max_length=8,
        pattern=USER_ID_PATTERN,
        description=(
            "Fictional customer ID in the exact format "
            "USR-1234."
        ),
    )

    @field_validator("user_id", mode="before")
    @classmethod
    def normalize_user_id(cls, value: Any) -> Any:
        if isinstance(value, str):
            return value.strip().upper()
        return value


class GetTransactionHistoryInput(StrictSchema):
    """Input schema for get_transaction_history."""

    user_id: str = Field(
        min_length=8,
        max_length=8,
        pattern=USER_ID_PATTERN,
        description=(
            "Fictional customer ID in the exact format "
            "USR-1234."
        ),
    )

    @field_validator("user_id", mode="before")
    @classmethod
    def normalize_user_id(cls, value: Any) -> Any:
        if isinstance(value, str):
            return value.strip().upper()
        return value


print("Input schemas defined.")

Input schemas defined.


## 5. Define output schemas

Each successful tool has its own output format.

All failures use a common `ToolErrorOutput` schema so the model receives a predictable error object rather than a raw Python exception.

In [5]:
class ToolErrorOutput(StrictSchema):
    """Standard output returned after a handled tool failure."""

    status: Literal["error"] = "error"
    tool_name: str
    error_code: Literal[
        "INVALID_TOOL_INPUT",
        "UNKNOWN_TOOL",
        "USER_NOT_FOUND",
        "UPSTREAM_TIMEOUT",
        "SERVICE_UNAVAILABLE",
        "INVALID_TOOL_OUTPUT",
        "DUPLICATE_CALL_PREVENTED",
        "UNEXPECTED_TOOL_ERROR",
    ]
    message: str
    retryable: bool


class CreditScoreSuccess(StrictSchema):
    """Successful credit-score output."""

    status: Literal["success"] = "success"
    user_id: str = Field(
        pattern=USER_ID_PATTERN,
    )
    credit_score: int = Field(
        ge=300,
        le=850,
    )
    rating: Literal[
        "POOR",
        "FAIR",
        "GOOD",
        "EXCELLENT",
    ]
    source: Literal["mock_credit_bureau"]
    retrieved_on: date


class Transaction(StrictSchema):
    """One fictional banking transaction."""

    transaction_id: str = Field(
        pattern=r"^TXN-[0-9]{5}$",
    )
    transaction_date: date
    description: str = Field(
        min_length=2,
        max_length=100,
    )
    category: Literal[
        "salary",
        "housing",
        "groceries",
        "utilities",
        "transport",
        "shopping",
        "investment",
        "other",
    ]
    transaction_type: Literal[
        "credit",
        "debit",
    ]
    amount: float = Field(gt=0)
    currency: Literal["INR"] = "INR"


class TransactionHistorySuccess(StrictSchema):
    """Successful transaction-history output."""

    status: Literal["success"] = "success"
    user_id: str = Field(
        pattern=USER_ID_PATTERN,
    )
    transactions: List[Transaction]
    total_transactions: int = Field(ge=0)
    total_credits: float = Field(ge=0)
    total_debits: float = Field(ge=0)
    currency: Literal["INR"] = "INR"
    source: Literal["mock_core_banking_system"]

    @model_validator(mode="after")
    def validate_aggregates(
        self,
    ) -> "TransactionHistorySuccess":
        if self.total_transactions != len(
            self.transactions
        ):
            raise ValueError(
                "total_transactions does not match "
                "the transaction list."
            )

        calculated_credits = round(
            sum(
                item.amount
                for item in self.transactions
                if item.transaction_type == "credit"
            ),
            2,
        )
        calculated_debits = round(
            sum(
                item.amount
                for item in self.transactions
                if item.transaction_type == "debit"
            ),
            2,
        )

        if round(self.total_credits, 2) != calculated_credits:
            raise ValueError(
                "total_credits does not match "
                "the transaction list."
            )

        if round(self.total_debits, 2) != calculated_debits:
            raise ValueError(
                "total_debits does not match "
                "the transaction list."
            )

        return self


CreditScoreOutput = Annotated[
    Union[
        CreditScoreSuccess,
        ToolErrorOutput,
    ],
    Field(discriminator="status"),
]

TransactionHistoryOutput = Annotated[
    Union[
        TransactionHistorySuccess,
        ToolErrorOutput,
    ],
    Field(discriminator="status"),
]

credit_output_adapter = TypeAdapter(
    CreditScoreOutput
)
transaction_output_adapter = TypeAdapter(
    TransactionHistoryOutput
)

print("Output schemas defined.")

Output schemas defined.


## 6. Inspect the JSON schemas

These are the schemas enforced by Pydantic. LangChain converts the input schemas into tool definitions supplied to the model.

In [6]:
print("GET CREDIT SCORE — INPUT SCHEMA")
pprint(GetCreditScoreInput.model_json_schema())

print("\nGET CREDIT SCORE — OUTPUT SCHEMA")
pprint(credit_output_adapter.json_schema())

print("\nGET TRANSACTION HISTORY — INPUT SCHEMA")
pprint(GetTransactionHistoryInput.model_json_schema())

print("\nGET TRANSACTION HISTORY — OUTPUT SCHEMA")
pprint(transaction_output_adapter.json_schema())

GET CREDIT SCORE — INPUT SCHEMA
{'additionalProperties': False,
 'description': 'Input schema for get_credit_score.',
 'properties': {'user_id': {'description': 'Fictional customer ID in the exact '
                                           'format USR-1234.',
                            'maxLength': 8,
                            'minLength': 8,
                            'pattern': '^USR-[0-9]{4}$',
                            'title': 'User Id',
                            'type': 'string'}},
 'required': ['user_id'],
 'title': 'GetCreditScoreInput',
 'type': 'object'}

GET CREDIT SCORE — OUTPUT SCHEMA
{'$defs': {'CreditScoreSuccess': {'additionalProperties': False,
                                  'description': 'Successful credit-score '
                                                 'output.',
                                  'properties': {'credit_score': {'maximum': 850,
                                                                  'minimum': 300,
                    

## 7. Create fictional banking data and simulated service errors

Special IDs are reserved for failure demonstrations:

| User ID | Simulated behaviour |
|---|---|
| `USR-5000` | Credit service timeout |
| `USR-5001` | Transaction service unavailable |
| `USR-9999` | Valid format, but user not found |

In [7]:
MOCK_CREDIT_DATABASE: Dict[str, int] = {
    "USR-1001": 782,
    "USR-1002": 668,
    "USR-1003": 541,
}


MOCK_TRANSACTION_DATABASE: Dict[
    str,
    List[Dict[str, Any]],
] = {
    "USR-1001": [
        {
            "transaction_id": "TXN-10001",
            "transaction_date": "2026-06-01",
            "description": "Monthly salary",
            "category": "salary",
            "transaction_type": "credit",
            "amount": 120000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-10002",
            "transaction_date": "2026-06-02",
            "description": "Apartment rent",
            "category": "housing",
            "transaction_type": "debit",
            "amount": 32000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-10003",
            "transaction_date": "2026-06-05",
            "description": "Supermarket purchase",
            "category": "groceries",
            "transaction_type": "debit",
            "amount": 6800.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-10004",
            "transaction_date": "2026-06-08",
            "description": "Mutual fund investment",
            "category": "investment",
            "transaction_type": "debit",
            "amount": 15000.00,
            "currency": "INR",
        },
    ],
    "USR-1002": [
        {
            "transaction_id": "TXN-20001",
            "transaction_date": "2026-06-01",
            "description": "Monthly salary",
            "category": "salary",
            "transaction_type": "credit",
            "amount": 85000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-20002",
            "transaction_date": "2026-06-03",
            "description": "Home loan payment",
            "category": "housing",
            "transaction_type": "debit",
            "amount": 28000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-20003",
            "transaction_date": "2026-06-07",
            "description": "Electricity and internet",
            "category": "utilities",
            "transaction_type": "debit",
            "amount": 5200.00,
            "currency": "INR",
        },
    ],
    "USR-1003": [
        {
            "transaction_id": "TXN-30001",
            "transaction_date": "2026-06-01",
            "description": "Monthly salary",
            "category": "salary",
            "transaction_type": "credit",
            "amount": 55000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-30002",
            "transaction_date": "2026-06-02",
            "description": "Apartment rent",
            "category": "housing",
            "transaction_type": "debit",
            "amount": 24000.00,
            "currency": "INR",
        },
        {
            "transaction_id": "TXN-30003",
            "transaction_date": "2026-06-06",
            "description": "Online shopping",
            "category": "shopping",
            "transaction_type": "debit",
            "amount": 12000.00,
            "currency": "INR",
        },
    ],
}


class UserNotFoundError(Exception):
    """Raised when a fictional user does not exist."""


class UpstreamTimeoutError(TimeoutError):
    """Raised to simulate an upstream timeout."""


class ServiceUnavailableError(ConnectionError):
    """Raised to simulate service unavailability."""


print("Mock data and service exceptions defined.")

Mock data and service exceptions defined.


## 8. Implement the two tools

The functions return JSON strings that conform to their declared success schemas.

Errors are intentionally raised here and converted into structured error outputs by the agent's safe execution layer.

In [8]:
def credit_rating(score: int) -> str:
    """Map a mock score to a simple rating."""
    if score >= 750:
        return "EXCELLENT"
    if score >= 670:
        return "GOOD"
    if score >= 580:
        return "FAIR"
    return "POOR"


@tool(args_schema=GetCreditScoreInput)
def get_credit_score(user_id: str) -> str:
    """
    Retrieve a fictional user's mock credit score.

    Use this tool when the user asks about credit score,
    credit standing, or a combined banking profile.
    """
    normalized_id = user_id.strip().upper()

    if normalized_id == "USR-5000":
        raise UpstreamTimeoutError(
            "The mock credit bureau timed out."
        )

    if normalized_id not in MOCK_CREDIT_DATABASE:
        raise UserNotFoundError(
            f"No credit record exists for {normalized_id}."
        )

    score = MOCK_CREDIT_DATABASE[normalized_id]

    output = CreditScoreSuccess(
        user_id=normalized_id,
        credit_score=score,
        rating=credit_rating(score),
        source="mock_credit_bureau",
        retrieved_on=date.today(),
    )

    return output.model_dump_json()


@tool(args_schema=GetTransactionHistoryInput)
def get_transaction_history(user_id: str) -> str:
    """
    Retrieve a fictional user's recent mock transactions.

    Use this tool when the user asks about transactions,
    credits, debits, spending, cash flow, or a combined
    banking profile.
    """
    normalized_id = user_id.strip().upper()

    if normalized_id == "USR-5001":
        raise ServiceUnavailableError(
            "The mock core-banking service is unavailable."
        )

    if normalized_id not in MOCK_TRANSACTION_DATABASE:
        raise UserNotFoundError(
            f"No transaction record exists for "
            f"{normalized_id}."
        )

    transactions = [
        Transaction.model_validate(item)
        for item in MOCK_TRANSACTION_DATABASE[
            normalized_id
        ]
    ]

    total_credits = round(
        sum(
            item.amount
            for item in transactions
            if item.transaction_type == "credit"
        ),
        2,
    )
    total_debits = round(
        sum(
            item.amount
            for item in transactions
            if item.transaction_type == "debit"
        ),
        2,
    )

    output = TransactionHistorySuccess(
        user_id=normalized_id,
        transactions=transactions,
        total_transactions=len(transactions),
        total_credits=total_credits,
        total_debits=total_debits,
        currency="INR",
        source="mock_core_banking_system",
    )

    return output.model_dump_json()


TOOLS: List[BaseTool] = [
    get_credit_score,
    get_transaction_history,
]

TOOL_REGISTRY: Dict[str, BaseTool] = {
    tool_object.name: tool_object
    for tool_object in TOOLS
}

OUTPUT_ADAPTERS: Dict[str, TypeAdapter] = {
    "get_credit_score": credit_output_adapter,
    "get_transaction_history": (
        transaction_output_adapter
    ),
}

print("Tools:", list(TOOL_REGISTRY))

Tools: ['get_credit_score', 'get_transaction_history']


## 9. Direct schema-validation examples

These tests do not require an OpenAI call.

The first payload is accepted and normalised. The following payloads are rejected:

- malformed user ID;
- missing required ID;
- undeclared extra field.

In [9]:
valid_input = GetCreditScoreInput.model_validate(
    {
        "user_id": " usr-1001 ",
    }
)

print("Accepted and normalised:")
pprint(valid_input.model_dump())

invalid_payloads = [
    {
        "user_id": "1001",
    },
    {},
    {
        "user_id": "USR-1001",
        "account_password": "must-not-be-accepted",
    },
]

for payload in invalid_payloads:
    print("\nTesting payload:")
    pprint(payload)

    try:
        GetCreditScoreInput.model_validate(payload)
        print("Unexpectedly accepted.")

    except ValidationError as exc:
        print("Rejected correctly:")
        print(exc)

Accepted and normalised:
{'user_id': 'USR-1001'}

Testing payload:
{'user_id': '1001'}
Rejected correctly:
1 validation error for GetCreditScoreInput
user_id
  String should have at least 8 characters [type=string_too_short, input_value='1001', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short

Testing payload:
{}
Rejected correctly:
1 validation error for GetCreditScoreInput
user_id
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

Testing payload:
{'account_password': 'must-not-be-accepted', 'user_id': 'USR-1001'}
Rejected correctly:
1 validation error for GetCreditScoreInput
account_password
  Extra inputs are not permitted [type=extra_forbidden, input_value='must-not-be-accepted', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


## 10. Implement the safe tool-execution layer

This function:

1. checks that the requested tool exists;
2. prevents identical repeated calls;
3. invokes the LangChain tool, triggering input validation;
4. catches known service failures;
5. parses and validates the tool output;
6. confirms that the returned user ID matches the requested ID; and
7. returns either validated success JSON or validated error JSON.

In [10]:
def build_tool_error(
    tool_name: str,
    error_code: str,
    message: str,
    retryable: bool,
) -> str:
    """Create a validated standard tool-error result."""
    error = ToolErrorOutput(
        tool_name=tool_name,
        error_code=error_code,
        message=message,
        retryable=retryable,
    )

    return error.model_dump_json()


def safe_execute_tool(
    tool_name: str,
    arguments: Dict[str, Any],
    executed_signatures: Set[str],
) -> str:
    """Execute and validate one model-selected tool call."""
    if tool_name not in TOOL_REGISTRY:
        return build_tool_error(
            tool_name=tool_name,
            error_code="UNKNOWN_TOOL",
            message=(
                f"The requested tool {tool_name!r} "
                "is not registered."
            ),
            retryable=False,
        )

    signature = json.dumps(
        {
            "tool_name": tool_name,
            "arguments": arguments,
        },
        sort_keys=True,
        default=str,
    )

    if signature in executed_signatures:
        return build_tool_error(
            tool_name=tool_name,
            error_code="DUPLICATE_CALL_PREVENTED",
            message=(
                "An identical tool call was already executed "
                "during this agent run."
            ),
            retryable=False,
        )

    executed_signatures.add(signature)
    selected_tool = TOOL_REGISTRY[tool_name]

    try:
        raw_output = selected_tool.invoke(arguments)

    except ValidationError as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="INVALID_TOOL_INPUT",
            message=(
                "Tool input failed schema validation: "
                + str(exc)
            ),
            retryable=False,
        )

    except UserNotFoundError as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="USER_NOT_FOUND",
            message=str(exc),
            retryable=False,
        )

    except UpstreamTimeoutError as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="UPSTREAM_TIMEOUT",
            message=str(exc),
            retryable=True,
        )

    except ServiceUnavailableError as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="SERVICE_UNAVAILABLE",
            message=str(exc),
            retryable=True,
        )

    except Exception as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="UNEXPECTED_TOOL_ERROR",
            message=(
                f"{type(exc).__name__}: {exc}"
            ),
            retryable=False,
        )

    try:
        parsed_output = json.loads(raw_output)

        validated_output = OUTPUT_ADAPTERS[
            tool_name
        ].validate_python(parsed_output)

        if (
            parsed_output.get("status") == "success"
            and "user_id" in arguments
            and parsed_output.get("user_id")
            != arguments["user_id"].strip().upper()
        ):
            raise ValueError(
                "Tool returned data for a different user ID."
            )

        return validated_output.model_dump_json()

    except Exception as exc:
        return build_tool_error(
            tool_name=tool_name,
            error_code="INVALID_TOOL_OUTPUT",
            message=(
                "Tool output failed schema validation: "
                f"{type(exc).__name__}: {exc}"
            ),
            retryable=False,
        )


print("Safe tool executor defined.")

Safe tool executor defined.


## 11. Reliably demonstrate handled tool failures

These tests run directly through the safe execution layer, so they work independently of which tools the model chooses.

In [11]:
failure_tests = [
    {
        "name": "Invalid input format",
        "tool_name": "get_credit_score",
        "arguments": {
            "user_id": "INVALID-ID",
        },
    },
    {
        "name": "Unknown customer",
        "tool_name": "get_credit_score",
        "arguments": {
            "user_id": "USR-9999",
        },
    },
    {
        "name": "Simulated timeout",
        "tool_name": "get_credit_score",
        "arguments": {
            "user_id": "USR-5000",
        },
    },
    {
        "name": "Simulated service outage",
        "tool_name": "get_transaction_history",
        "arguments": {
            "user_id": "USR-5001",
        },
    },
    {
        "name": "Unknown tool",
        "tool_name": "get_account_password",
        "arguments": {
            "user_id": "USR-1001",
        },
    },
]

failure_rows = []

for test in failure_tests:
    output_json = safe_execute_tool(
        tool_name=test["tool_name"],
        arguments=test["arguments"],
        executed_signatures=set(),
    )
    output = json.loads(output_json)

    failure_rows.append(
        {
            "Test": test["name"],
            "Tool": test["tool_name"],
            "Status": output["status"],
            "Error code": output.get(
                "error_code"
            ),
            "Retryable": output.get(
                "retryable"
            ),
            "Message": output.get("message"),
        }
    )

display(pd.DataFrame(failure_rows))

,Test,Tool,Status,Error code,Retryable,Message
0,Invalid input format,get_credit_score,error,INVALID_TOOL_INPUT,False,Tool input failed schema validation: 1 validat...
1,Unknown customer,get_credit_score,error,USER_NOT_FOUND,False,No credit record exists for USR-9999.
2,Simulated timeout,get_credit_score,error,UPSTREAM_TIMEOUT,True,The mock credit bureau timed out.
3,Simulated service outage,get_transaction_history,error,SERVICE_UNAVAILABLE,True,The mock core-banking service is unavailable.
4,Unknown tool,get_account_password,error,UNKNOWN_TOOL,False,The requested tool 'get_account_password' is n...


## 12. Demonstrate output-schema rejection

This deliberately broken function returns an impossible credit score. The output validator rejects it.

The malformed tool is used only for this isolated validation demonstration and is not exposed to the agent.

In [12]:
def malformed_credit_tool_output() -> str:
    """Return deliberately invalid output."""
    return json.dumps(
        {
            "status": "success",
            "user_id": "USR-1001",
            "credit_score": 999,
            "rating": "EXCELLENT",
            "source": "mock_credit_bureau",
            "retrieved_on": str(date.today()),
        }
    )


try:
    credit_output_adapter.validate_json(
        malformed_credit_tool_output()
    )
    print("Unexpectedly accepted malformed output.")

except ValidationError as exc:
    print("Malformed output rejected correctly:")
    print(exc)

Malformed output rejected correctly:
1 validation error for tagged-union[CreditScoreSuccess,ToolErrorOutput]
success.credit_score
  Input should be less than or equal to 850 [type=less_than_equal, input_value=999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 13. Create the dynamically selecting banking agent

`ChatOpenAI.bind_tools()` supplies both tool schemas to the model.

The model decides dynamically:

- credit-related request → `get_credit_score`;
- transaction/spending request → `get_transaction_history`;
- combined banking-profile request → both tools.

The agent loop executes every returned tool call and adds a corresponding `ToolMessage`, then allows the model to produce a grounded final response.

In [13]:
BANKING_AGENT_INSTRUCTIONS = """
You are a banking-information assistant operating only on
fictional mock data in an educational notebook.

Available tools:
- get_credit_score: use for credit score, credit standing, or
  credit-related questions.
- get_transaction_history: use for transactions, spending,
  credits, debits, or cash-flow questions.

Tool-selection rules:
1. Select tools dynamically according to the user's request.
2. If the user asks for a combined banking profile, use both.
3. Never invent a credit score or transaction.
4. A valid fictional user ID has the exact form USR-1234.
5. Do not ask for or expose passwords, PINs, OTPs, complete
   account numbers, or real banking credentials.
6. Read structured tool errors carefully. Explain failures
   clearly and do not claim success when a tool failed.
7. Do not make lending, investment, or eligibility decisions.
8. Keep the final response concise and clearly state that the
   data is fictional.
"""


class BankingToolAgent:
    """A small explicit LangChain tool-calling agent."""

    def __init__(
        self,
        model: str = MODEL,
        max_steps: int = 4,
    ):
        self.model_name = model
        self.max_steps = max_steps

        self.model = ChatOpenAI(
            model=model,
            temperature=0,
        )

        self.model_with_tools = self.model.bind_tools(
            TOOLS,
            tool_choice="auto",
            strict=True,
        )

    def run(
        self,
        user_request: str,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        """Run until the model returns text or step limit ends."""
        messages = [
            SystemMessage(
                content=BANKING_AGENT_INSTRUCTIONS
            ),
            HumanMessage(content=user_request),
        ]

        executed_signatures: Set[str] = set()
        trace: List[Dict[str, Any]] = []
        selected_tools: List[str] = []

        for step in range(1, self.max_steps + 1):
            ai_message = self.model_with_tools.invoke(
                messages
            )
            messages.append(ai_message)

            tool_calls = ai_message.tool_calls or []

            if verbose:
                print(
                    f"\n--- Agent step {step} ---"
                )

            if not tool_calls:
                final_text = (
                    ai_message.content
                    if isinstance(
                        ai_message.content,
                        str,
                    )
                    else str(ai_message.content)
                )

                trace.append(
                    {
                        "step": step,
                        "event": "FINAL_RESPONSE",
                        "content": final_text,
                    }
                )

                return {
                    "final_response": final_text,
                    "selected_tools": selected_tools,
                    "trace": trace,
                    "termination_reason": (
                        "MODEL_RETURNED_FINAL_RESPONSE"
                    ),
                }

            if verbose:
                print(
                    "Model selected:",
                    [
                        call["name"]
                        for call in tool_calls
                    ],
                )

            for tool_call in tool_calls:
                tool_name = tool_call["name"]
                arguments = tool_call["args"]
                tool_call_id = tool_call["id"]

                selected_tools.append(tool_name)

                result_json = safe_execute_tool(
                    tool_name=tool_name,
                    arguments=arguments,
                    executed_signatures=(
                        executed_signatures
                    ),
                )

                result_object = json.loads(
                    result_json
                )

                trace.append(
                    {
                        "step": step,
                        "event": "TOOL_EXECUTED",
                        "tool_name": tool_name,
                        "arguments": arguments,
                        "result": result_object,
                    }
                )

                if verbose:
                    print(
                        f"Tool: {tool_name}"
                    )
                    print("Arguments:")
                    pprint(arguments)
                    print("Validated result:")
                    pprint(result_object)

                messages.append(
                    ToolMessage(
                        content=result_json,
                        tool_call_id=tool_call_id,
                        name=tool_name,
                    )
                )

        fallback_message = (
            "The agent stopped after reaching its maximum "
            "number of steps. No further tools were called."
        )

        trace.append(
            {
                "step": self.max_steps,
                "event": "STEP_LIMIT_REACHED",
                "content": fallback_message,
            }
        )

        return {
            "final_response": fallback_message,
            "selected_tools": selected_tools,
            "trace": trace,
            "termination_reason": (
                "MAX_AGENT_STEPS_REACHED"
            ),
        }


banking_agent = BankingToolAgent()
print("BankingToolAgent created.")

BankingToolAgent created.


## 14. Dynamic selection test A — Credit score only

The request mentions only credit information, so the model should normally select only `get_credit_score`.

In [14]:
credit_only_result = banking_agent.run(
    "Retrieve the fictional credit score for USR-1001."
)

print("\nSelected tools:")
print(credit_only_result["selected_tools"])

print("\nFinal response:")
print(credit_only_result["final_response"])


--- Agent step 1 ---
Model selected: ['get_credit_score']
Tool: get_credit_score
Arguments:
{'user_id': 'USR-1001'}
Validated result:
{'credit_score': 782,
 'rating': 'EXCELLENT',
 'retrieved_on': '2026-07-12',
 'source': 'mock_credit_bureau',
 'status': 'success',
 'user_id': 'USR-1001'}

--- Agent step 2 ---

Selected tools:
['get_credit_score']

Final response:
The fictional credit score for user ID USR-1001 is 782, which is rated as "EXCELLENT." This data is from a mock credit bureau and was retrieved on July 12, 2026.


## 15. Dynamic selection test B — Transaction history only

The request concerns spending and transactions, so the model should normally select `get_transaction_history`.

In [15]:
transactions_only_result = banking_agent.run(
    "Show a concise summary of recent fictional "
    "transactions and spending for USR-1002."
)

print("\nSelected tools:")
print(transactions_only_result["selected_tools"])

print("\nFinal response:")
print(
    transactions_only_result["final_response"]
)


--- Agent step 1 ---
Model selected: ['get_transaction_history']
Tool: get_transaction_history
Arguments:
{'user_id': 'USR-1002'}
Validated result:
{'currency': 'INR',
 'source': 'mock_core_banking_system',
 'status': 'success',
 'total_credits': 85000.0,
 'total_debits': 33200.0,
 'total_transactions': 3,
 'transactions': [{'amount': 85000.0,
                   'category': 'salary',
                   'currency': 'INR',
                   'description': 'Monthly salary',
                   'transaction_date': '2026-06-01',
                   'transaction_id': 'TXN-20001',
                   'transaction_type': 'credit'},
                  {'amount': 28000.0,
                   'category': 'housing',
                   'currency': 'INR',
                   'description': 'Home loan payment',
                   'transaction_date': '2026-06-03',
                   'transaction_id': 'TXN-20002',
                   'transaction_type': 'debit'},
                  {'amount': 5200.0,
       

## 16. Dynamic selection test C — Combined profile

A combined request requires both tools.

In [16]:
combined_result = banking_agent.run(
    "Using the fictional records for USR-1001, "
    "summarise both credit standing and recent "
    "transaction activity. Do not make a lending decision."
)

print("\nSelected tools:")
print(combined_result["selected_tools"])

print("\nFinal response:")
print(combined_result["final_response"])


--- Agent step 1 ---
Model selected: ['get_credit_score', 'get_transaction_history']
Tool: get_credit_score
Arguments:
{'user_id': 'USR-1001'}
Validated result:
{'credit_score': 782,
 'rating': 'EXCELLENT',
 'retrieved_on': '2026-07-12',
 'source': 'mock_credit_bureau',
 'status': 'success',
 'user_id': 'USR-1001'}
Tool: get_transaction_history
Arguments:
{'user_id': 'USR-1001'}
Validated result:
{'currency': 'INR',
 'source': 'mock_core_banking_system',
 'status': 'success',
 'total_credits': 120000.0,
 'total_debits': 53800.0,
 'total_transactions': 4,
 'transactions': [{'amount': 120000.0,
                   'category': 'salary',
                   'currency': 'INR',
                   'description': 'Monthly salary',
                   'transaction_date': '2026-06-01',
                   'transaction_id': 'TXN-10001',
                   'transaction_type': 'credit'},
                  {'amount': 32000.0,
                   'category': 'housing',
                   'currency': 'INR

## 17. Agent-level invalid-input demonstration

The model may attempt to call a tool with the malformed ID. The safe execution layer rejects it, returns a structured error, and allows the model to explain the problem rather than crashing.

In [17]:
invalid_agent_result = banking_agent.run(
    "Get the credit score for customer ABC123."
)

print("\nSelected tools:")
print(invalid_agent_result["selected_tools"])

print("\nFinal response:")
print(invalid_agent_result["final_response"])


--- Agent step 1 ---

Selected tools:
[]

Final response:
The customer ID you provided is not in the correct format. Please provide a valid fictional customer ID in the format USR-1234.


## 18. Agent-level service-failure demonstration

`USR-5000` has a valid schema but triggers a simulated upstream credit-service timeout.

In [18]:
failure_agent_result = banking_agent.run(
    "Retrieve the fictional credit score for USR-5000."
)

print("\nSelected tools:")
print(failure_agent_result["selected_tools"])

print("\nFinal response:")
print(failure_agent_result["final_response"])


--- Agent step 1 ---

Selected tools:
[]

Final response:
The user ID you provided, USR-5000, is not in the correct format. Please ensure the user ID follows the format USR-1234.


## 19. Inspect an execution trace

The trace captures model-selected tools, validated arguments, successful outputs, and handled failures.

In [19]:
trace_rows = []

for event in combined_result["trace"]:
    trace_rows.append(
        {
            "Step": event.get("step"),
            "Event": event.get("event"),
            "Tool": event.get("tool_name"),
            "Arguments": event.get("arguments"),
            "Result status": (
                event.get("result", {}).get(
                    "status"
                )
                if isinstance(
                    event.get("result"),
                    dict,
                )
                else None
            ),
            "Error code": (
                event.get("result", {}).get(
                    "error_code"
                )
                if isinstance(
                    event.get("result"),
                    dict,
                )
                else None
            ),
        }
    )

display(pd.DataFrame(trace_rows))

,Step,Event,Tool,Arguments,Result status,Error code
0,1,TOOL_EXECUTED,get_credit_score,{'user_id': 'USR-1001'},success,None
1,1,TOOL_EXECUTED,get_transaction_history,{'user_id': 'USR-1001'},success,None
2,2,FINAL_RESPONSE,NaN,None,NaN,None


## 20. Optional direct tool calls

These calls are useful for testing the tools independently of the model.

In [20]:
direct_signatures: Set[str] = set()

print("Direct credit result:")
pprint(
    json.loads(
        safe_execute_tool(
            tool_name="get_credit_score",
            arguments={
                "user_id": "USR-1002",
            },
            executed_signatures=(
                direct_signatures
            ),
        )
    )
)

print("\nDirect transaction result:")
pprint(
    json.loads(
        safe_execute_tool(
            tool_name=(
                "get_transaction_history"
            ),
            arguments={
                "user_id": "USR-1002",
            },
            executed_signatures=(
                direct_signatures
            ),
        )
    )
)

Direct credit result:
{'credit_score': 668,
 'rating': 'FAIR',
 'retrieved_on': '2026-07-12',
 'source': 'mock_credit_bureau',
 'status': 'success',
 'user_id': 'USR-1002'}

Direct transaction result:
{'currency': 'INR',
 'source': 'mock_core_banking_system',
 'status': 'success',
 'total_credits': 85000.0,
 'total_debits': 33200.0,
 'total_transactions': 3,
 'transactions': [{'amount': 85000.0,
                   'category': 'salary',
                   'currency': 'INR',
                   'description': 'Monthly salary',
                   'transaction_date': '2026-06-01',
                   'transaction_id': 'TXN-20001',
                   'transaction_type': 'credit'},
                  {'amount': 28000.0,
                   'category': 'housing',
                   'currency': 'INR',
                   'description': 'Home loan payment',
                   'transaction_date': '2026-06-03',
                   'transaction_id': 'TXN-20002',
                   'transaction_type': 'd

## Requirements mapping

| Coding requirement | Implementation |
|---|---|
| Define credit-score input schema | `GetCreditScoreInput` |
| Define transaction-history input schema | `GetTransactionHistoryInput` |
| Define credit-score output schema | `CreditScoreSuccess` + `ToolErrorOutput` |
| Define transaction output schema | `TransactionHistorySuccess` + `ToolErrorOutput` |
| Reject invalid inputs | Pydantic pattern, required-field, and extra-field validation |
| Validate tool outputs | `TypeAdapter` plus aggregate checks |
| Select tools dynamically | `ChatOpenAI.bind_tools(..., tool_choice="auto")` |
| Execute multiple selected tools | Explicit loop over `ai_message.tool_calls` |
| Handle unknown users | `USER_NOT_FOUND` |
| Handle timeout | `UPSTREAM_TIMEOUT` |
| Handle service outage | `SERVICE_UNAVAILABLE` |
| Handle malformed output | `INVALID_TOOL_OUTPUT` |
| Prevent repeated identical calls | `DUPLICATE_CALL_PREVENTED` |
| Prevent endless agent execution | `max_steps` |
| Trace tool execution | Agent `trace` |
| Use fictional data only | Mock databases and synthetic IDs |

## Key learning

Tool calling is reliable only when validation surrounds both sides of the tool boundary:

```text
Model-generated arguments
        ↓
Input validation
        ↓
Tool execution
        ↓
Output validation
        ↓
Model-visible ToolMessage
```

A robust agent should never pass raw model arguments directly into sensitive systems and should never trust unvalidated tool output.

## Limitations and production considerations

This is an educational mock system. A real banking implementation would additionally require:

- authenticated and authorised tool access;
- field-level access controls;
- encryption in transit and at rest;
- audit logging without leaking sensitive data;
- secret management;
- consent and data-retention controls;
- rate limiting and circuit breakers;
- idempotency controls;
- provider-specific retries and timeouts;
- monitoring and alerting;
- regulatory, privacy, security, and legal review;
- human review for consequential financial decisions.

## References

- LangChain tools:  
  https://docs.langchain.com/oss/python/langchain/tools

- LangChain agents and dynamic tools:  
  https://docs.langchain.com/oss/python/langchain/agents

- LangChain OpenAI tool binding:  
  https://docs.langchain.com/oss/python/integrations/chat/openai

- LangChain structured output:  
  https://docs.langchain.com/oss/python/langchain/structured-output

- OpenAI function calling:  
  https://developers.openai.com/api/docs/guides/function-calling